# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Traffic metrics are classically heavy-tailed here: `impressions_90d` has skew ~11.4 (median 731,
mean 5,200, max 517,715) and `clicks_90d` skew ~18.3 (median 1, mean 16, max 4,178) -- a handful
of giant pages, a long tail of near-zero pages. `word_count` is comparatively tame (skew ~0.9).
Every test below either uses grouped medians / weighted rates (weighting by the true denominator,
not averaging per-row rates) instead of a plain mean on the raw totals -- a plain mean or Pearson
correlation on `impressions_90d` as-is would be dominated by the handful of giant pages.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

for col in ["impressions_90d", "clicks_90d", "sessions_90d", "word_count"]:
    print(f"{col:<16} skew={df[col].skew():6.1f}  median={df[col].median():>10,.0f}  "
          f"mean={df[col].mean():>10,.1f}  max={df[col].max():>10,.0f}")


impressions_90d  skew=  11.4  median=       731  mean=   5,200.4  max=   517,715
clicks_90d       skew=  18.3  median=         1  mean=      16.1  max=     4,178
sessions_90d     skew=  12.1  median=         7  mean=      37.1  max=     4,345
word_count       skew=   0.9  median=     2,877  mean=   3,107.8  max=     9,546


## 2. Signal test #1 / #2 / #3 (verdict each)

**Test 1 -- "Stale pages get less engagement than fresh pages."**
Weighted engagement rate (`engaged_sessions_90d`/`sessions_90d`, summed not averaged) by
`freshness_tier`, restricted to pages with real sessions: 0-30d = 3.00% (n=20,480), 31-90d = 2.98%
(n=175), 91-180d = 2.41% (n=9,171), 181+d = 2.70% (n=174). **Verdict: MIXED.** The two big buckets
(0-30 vs 91-180, both well above the sample floor) support the claim -- freshest pages engage
noticeably better. But the most-stale bucket (181+) doesn't continue the trend, and its n=174 is
thin enough relative to the 20k+ freshest bucket that I don't trust that one data point to overturn
the direction. MIXED, not CONFIRMED, is the honest read.

**Test 2 -- "Worse search position means lower CTR."**
Weighted CTR (`clicks_90d`/`impressions_90d`, summed) by `position_tier`, real position data only:
top_3 = 0.49% (n=1,116), page_1 = 0.35% (n=11,814), striking = 0.35% (n=7,304), page_3_5 = 0.15%
(n=7,242), deep = 0.04% (n=1,319). **Verdict: CONFIRMED.** CTR falls monotonically (page_1 and
striking are effectively tied) as position worsens, every bucket comfortably above the sample floor
-- the CTR cliff from the Week-1 notebook holds up as a real, well-populated pattern.

**Test 3 -- "Longer content gets more search demand."**
Median `impressions_90d` by `word_count_tier`: <1000 words = 4 (n=973), 1000-2000 = 172 (n=3,780),
2000-3500 = 997 (n=11,263), 3500+ = 1,340 (n=6,285). **Verdict: CONFIRMED, but read it as
association, not proof of a lever.** Impressions climb monotonically with length, all buckets well
above the floor -- but this is length correlating with demand, likely because longer pieces target
bigger, more competitive topics in the first place, not evidence that lengthening a specific page
would grow its own traffic. That distinction matters for what a content team does with this finding
(section 4).


In [2]:
eng = df[df["sessions_90d"] > 0]
t1 = eng.groupby("freshness_tier").apply(lambda x: pd.Series({
    "n": len(x),
    "weighted_engagement_rate_pct": x["engaged_sessions_90d"].sum() / x["sessions_90d"].sum() * 100,
}))
print("Test 1 -- freshness_tier vs weighted engagement rate:")
print(t1.round(2))

pos = df[df["avg_position"] > 0]
t2 = pos.groupby("position_tier").apply(lambda x: pd.Series({
    "n": len(x),
    "weighted_ctr_pct": x["clicks_90d"].sum() / x["impressions_90d"].sum() * 100,
})).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("\nTest 2 -- position_tier vs weighted CTR:")
print(t2.round(2))

wc = df.dropna(subset=["word_count_tier"])
t3 = wc.groupby("word_count_tier").apply(lambda x: pd.Series({
    "n": len(x),
    "median_impressions_90d": x["impressions_90d"].median(),
})).reindex(["<1000", "1000-2000", "2000-3500", "3500+"])
print("\nTest 3 -- word_count_tier vs median impressions:")
print(t3)


Test 1 -- freshness_tier vs weighted engagement rate:
                      n  weighted_engagement_rate_pct
freshness_tier                                       
0-30            20480.0                          3.00
181+              174.0                          2.70
31-90             175.0                          2.98
91-180           9171.0                          2.41

Test 2 -- position_tier vs weighted CTR:
                     n  weighted_ctr_pct
position_tier                           
top_3           1116.0              0.49
page_1         11814.0              0.35
striking        7304.0              0.35
page_3_5        7242.0              0.15
deep            1319.0              0.04

Test 3 -- word_count_tier vs median impressions:
                       n  median_impressions_90d
word_count_tier                                 
<1000              973.0                     4.0
1000-2000         3780.0                   172.0
2000-3500        11263.0                   997.

## 3. The flag-linked test

FlyRank's real `stale_visible_page` reason code (also used in my Week-4 baseline) assumes that
pages which are both stale AND still visible are disproportionately likely to be declining --
that's the whole justification for surfacing them first. Testing that assumption directly against
the observed decline rate:

At the strict threshold the reference pipeline uses (`days_since_last_update >= 180` and
`impressions_90d >= 500`), n=17 -- **below the ~50-row floor, so no verdict**: "insufficient data"
is the honest answer, not a headline number from 17 rows. At the looser, still-defensible threshold
my own Week-4 baseline actually uses (`>= 90` days, `>= 250` impressions), n=7,430 -- comfortably
above the floor. Decline rate among those pages: 61.9%, versus a 54.2% base rate overall.

**Verdict: CONFIRMED, but modestly, not dramatically.** The flag's assumption holds directionally
-- stale+visible pages really are somewhat more likely to be declining than a random page -- but
the lift is about 8 points, not a night-and-day difference. That's exactly why the Week-4 baseline
needed two more signals (CTR gap, engagement) stacked on top of "stale + visible" to reach a useful
precision@20, and why a model with access to more signals (Week 5) has real room to improve.


In [3]:
base_rate = df["trend_direction"].str.lower().eq("down").mean()

strict = df[(df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)]
print(f"Strict threshold (180d, 500 impr): n={len(strict)}  ", end="")
print("insufficient data (< 50-row floor)" if len(strict) < 50
      else f"decline rate={strict['trend_direction'].str.lower().eq('down').mean():.3f}")

loose = df[(df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= 250)]
loose_rate = loose["trend_direction"].str.lower().eq("down").mean()
print(f"Looser threshold (90d, 250 impr):  n={len(loose)}  decline rate={loose_rate:.3f}")
print(f"Base decline rate (all pages):     n={len(df)}  decline rate={base_rate:.3f}")
print(f"\nLift: {loose_rate - base_rate:+.3f} ({(loose_rate/base_rate - 1)*100:+.1f}%)")


Strict threshold (180d, 500 impr): n=17  insufficient data (< 50-row floor)
Looser threshold (90d, 250 impr):  n=7430  decline rate=0.619
Base decline rate (all pages):     n=30000  decline rate=0.542

Lift: +0.077 (+14.1%)


## 4. What this means in practice

Two things a content team should act on: (1) position and CTR are the strongest, most reliable
levers here -- the CTR cliff by position (Test 2) is large, monotonic, and well-populated, so
title/snippet work on page-1-adjacent, low-CTR pages is a safe bet. (2) "Stale and visible" is a
real but modest signal on its own (an 8-point lift over base rate) -- treat it as one input to
stack with others (as the Week-4 baseline does), not a rule to act on alone. And be careful with
the word-count finding: longer pages showing more demand is real, but it's an association across
different topics, not proof that lengthening any one page would grow its traffic -- don't sell
"just add more words" as the takeaway.


In [4]:
def floor_check(n, floor=50):
    return "OK (n >= floor)" if n >= floor else "INSUFFICIENT DATA (n < floor)"

for label, n in [
    ("Test 1, thinnest bucket (181+)", 174),
    ("Test 2, thinnest bucket (top_3)", 1116),
    ("Test 3, thinnest bucket (<1000)", 973),
    ("Flag test, strict threshold", 17),
    ("Flag test, loose threshold", 7430),
]:
    print(f"{label:<34} n={n:<6} -> {floor_check(n)}")


Test 1, thinnest bucket (181+)     n=174    -> OK (n >= floor)
Test 2, thinnest bucket (top_3)    n=1116   -> OK (n >= floor)
Test 3, thinnest bucket (<1000)    n=973    -> OK (n >= floor)
Flag test, strict threshold        n=17     -> INSUFFICIENT DATA (n < floor)
Flag test, loose threshold         n=7430   -> OK (n >= floor)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.